<a href="https://colab.research.google.com/github/Malang43/RAG-4bit-Unsloth/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers==5.2.0 accelerate==1.12.0 bitsandbytes==0.49.2 torch faiss-cpu sentence-transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.1 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
model_name = "unsloth/Qwen3-8B-Base-unsloth-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model (4-bit handled internally by repo)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically uses GPU if available
    torch_dtype=torch.float16
)

# Pipeline for easy generation
rag_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.75G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

In [4]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # small & fast
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = embed_model.to(device)

def embed_text(texts):
    embeddings = []
    for txt in texts:
        vec = embed_model.encode(txt, convert_to_numpy=True, device=device)
        embeddings.append(vec)
    return np.vstack(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
docs = [
    "MRI scans are covered under the premium medical insurance plan.",
    "Emergency hospitalization is fully covered under the gold plan.",
    "Dental check-ups are partially covered under the standard plan."
]


In [6]:
chunks = docs

In [7]:
embeddings = embed_text(chunks)
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)
print("FAISS index built with", index.ntotal, "vectors")

FAISS index built with 3 vectors


In [8]:
def retrieve(query, k=2):
    q_vec = embed_text([query])
    distances, idxs = index.search(q_vec, k)
    results = [chunks[i] for i in idxs[0]]
    return " ".join(results)

In [9]:
def rag_generate(query):
    context = retrieve(query)
    prompt = f"""
Use ONLY the information from the context below.

Context:
{context}

Question:
{query}

Answer strictly from the context:
"""
    output = rag_pipe(prompt, max_new_tokens=150, temperature=0.2)
    return output[0]['generated_text']

In [10]:
query = "Does insurance cover MRI scans?"
answer = rag_generate(query)
print("Query:", query)
print("Answer:", answer)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Does insurance cover MRI scans?
Answer: 
Use ONLY the information from the context below.

Context:
MRI scans are covered under the premium medical insurance plan. Dental check-ups are partially covered under the standard plan.

Question:
Does insurance cover MRI scans?

Answer strictly from the context:
Yes, insurance covers MRI scans under the premium medical insurance plan.
